In [ ]:
# Percentage change in mean ridership: pre -> post
pct_change = pd.DataFrame({
    "Pre Mean": pre_stats["Mean"],
    "Post Mean": post_stats["Mean"],
})
pct_change["Change"] = pct_change["Post Mean"] - pct_change["Pre Mean"]
pct_change["% Change"] = (pct_change["Change"] / pct_change["Pre Mean"] * 100).round(1)

# Add treatment flag
pct_change["Group"] = pct_change.index.map(
    lambda x: "Treatment" if x == "gold" else "Control"
)
display(pct_change.sort_values("% Change", ascending=False))

Compute the percentage change in mean ridership from pre to post for each line. This gives a quick view of which lines gained or lost riders after March 2016.

In [ ]:
# Summary stats by line and period
pre_df = analysis_df[analysis_df["period"] == "pre"]
post_df = analysis_df[analysis_df["period"] == "post"]

print("=== PRE-EXTENSION (Jan 2012 – Feb 2016) ===")
pre_stats = summary_stats_table(pre_df, "line_id", "avg_daily_boardings")
display(pre_stats.round(0))

print("\n=== POST-EXTENSION (Apr 2016 – Dec 2019) ===")
post_stats = summary_stats_table(post_df, "line_id", "avg_daily_boardings")
display(post_stats.round(0))

### 1.4 Summary Statistics — Pre vs Post Extension

This table shows descriptive statistics (mean, median, std, IQR) for each line's weekday boardings, broken out by pre-extension (Jan 2012 – Feb 2016) and post-extension (Apr 2016 – Dec 2019) periods.

In [ ]:
fig = plot_ridership_trends(
    ridership,
    line_ids=["gold", "a_line", "red", "green", "expo", "a_line_merged"],
    end_date=None,
    title="Monthly Avg. Weekday Boardings — Full Timeline (2009–2025)",
)
plt.show()

### 1.3 Full Timeline Including COVID and Recovery (2009–2025)

For context, here is the full data range including the COVID disruption (2020–2021) and recovery. The primary analysis excludes this period, but it informs our robustness checks later.

In [ ]:
fig = plot_treatment_vs_control(
    ridership,
    end_date="2019-12-31",
    title="Treatment (Gold Line) vs Control Lines — Avg. Weekday Boardings",
)
plt.show()

### 1.2 Treatment vs Control — Aggregated View

To visualize the DiD design, we plot the Gold Line (treatment) against the average of all control lines. If the extension had a causal effect, we should see the Gold Line diverge from the control group trend after March 2016.

In [ ]:
fig = plot_ridership_trends(
    ridership,
    line_ids=["gold", "a_line", "red", "green", "expo"],
    end_date="2019-12-31",
    title="Monthly Avg. Weekday Boardings by Line (2009–2019)",
)
plt.show()

---
## 1. Exploratory Data Analysis

### 1.1 Ridership Trends Over Time

The first visualization shows monthly average weekday boardings for each rail line from 2009 to 2019. The Gold Line (treatment) is highlighted. A vertical dashed line marks the Foothill Extension opening in March 2016.

We expect to see: (a) a general upward trend in system ridership leading up to 2016, (b) a visible bump or level shift in Gold Line ridership after the extension, and (c) relatively stable or declining trends in control lines for comparison.

In [ ]:
# Primary analysis window: 2012-01 to 2019-12 (pre-COVID)
ANALYSIS_START = "2012-01-01"
ANALYSIS_END = "2019-12-31"
EXTENSION_DATE = pd.Timestamp("2016-03-05")

analysis_df = ridership[
    (ridership["date"] >= ANALYSIS_START) & (ridership["date"] <= ANALYSIS_END)
].copy()

# For primary DiD: gold = treatment, other established lines = control
# Exclude k_line (opened 2022) and a_line_merged (post-Regional Connector)
PRIMARY_LINES = ["gold", "a_line", "red", "green", "expo"]
analysis_df = analysis_df[analysis_df["line_id"].isin(PRIMARY_LINES)].copy()

# Create a simpler treatment label for plotting
analysis_df["group"] = analysis_df["line_id"].apply(
    lambda x: "Treatment (Gold)" if x == "gold" else "Control"
)

print(f"Analysis window: {ANALYSIS_START} to {ANALYSIS_END}")
print(f"Rows: {len(analysis_df)}")
print(f"Lines: {sorted(analysis_df['line_id'].unique())}")
print(f"Pre-extension rows: {(analysis_df['period'] == 'pre').sum()}")
print(f"Post-extension rows: {(analysis_df['period'] == 'post').sum()}")

Define the primary analysis window. We focus on January 2012 – December 2019 to have at least 4 years of pre-period data and to avoid COVID-era disruption. The Gold Line data begins in 2012 (line opened 2003, but Streets For All data for Gold starts 2012).

In [ ]:
ridership = pd.read_csv(project_root / "data" / "clean" / "ridership_clean.csv", parse_dates=["date"])
stations = pd.read_csv(project_root / "data" / "clean" / "station_locations_clean.csv", parse_dates=["opened_date"])

print(f"Ridership: {len(ridership)} rows, {ridership['date'].min().date()} to {ridership['date'].max().date()}")
print(f"Lines: {sorted(ridership['line_id'].unique())}")
print(f"Stations: {len(stations)} total, {stations['is_foothill_2016'].sum()} treatment")
ridership.head()

Load the cleaned ridership and station location data produced by Phase 1. The ridership data has 944 rows covering 6 rail lines from 2009–2025, with treatment/control and pre/post period flags already assigned.

In [ ]:
import sys
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import src modules
project_root = pathlib.Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.viz import (
    set_default_theme,
    plot_ridership_trends,
    plot_treatment_vs_control,
    plot_distributions,
    plot_boxplot_comparison,
)
from src.stats import (
    summary_stats_table,
    shapiro_wilk_test,
    independent_ttest,
    mann_whitney_u,
)

set_default_theme()
print("Libraries loaded.")

## Setup and Data Loading

# Impact of the Gold Line Foothill Extension on LA Metro Ridership

**Research Question:** Did the March 2016 opening of the Gold Line Foothill Extension Phase 2A lead to a statistically significant change in ridership on the Gold Line compared to other Metro rail lines?

**Method:** Difference-in-differences (DiD) analysis comparing Gold Line (treatment) ridership trends against other Metro rail lines (control) before and after the extension opening.

**Data:** Monthly average weekday boardings by line, January 2009 – June 2025, from the Streets For All ridership dashboard.